# Lab 5. Flask - serwowanie modelu przez REST API

Flask  - pythonowy webowy famework
MLflow

## 1. Pierwsze API
- budujemy serwer korzystając z gotowca
- podobny schemat w licznych językach programowania

In [4]:
%%file app.py
from flask import Flask

app = Flask(__name__)

# ROUTING - sciezki do ktorych użytkownik ma dostęp
@app.route("/")                      # adres URL: http://localhost:5000/
def home():
    return "Witaj w systemie monitoringu transakcji!"

if __name__ == "__main__":              # Funkcja nie zadziała, jeżli kod zostanie zaimportowany
    app.run(host="0.0.0.0", port=5000)  # Uruchamianie na konkretnych porcie

Writing app.py


Podejście to vs podejście klasyczne (nie muszą istnieć podkatalogi)

In [5]:
# dekorator - wewnątrz funkcji uruchamiamy inną funkcję
def dekorator(fun):
    print("Wpisz tresc z dekoratora")
    return fun()

def test():
    print("test")


In [6]:
dekorator(test)

Wpisz tresc z dekoratora
test


In [7]:
@dekorator
def test2():
    print("test2")

Wpisz tresc z dekoratora
test2


Funkcja została zdefiniowana i wykonanna przez dekorator

Odpytujemy serwer - z poziomu Pythona

In [8]:
import requests

In [9]:
response = requests.get("http://localhost:5000/")

In [10]:
response

<Response [200]>

In [11]:
response.text

'Witaj w systemie monitoringu transakcji!'

My będziemy pól tekstowych unikać, raczej będziemy korzystać z JSON

In [12]:
response.status_code

200

Zamykamy nasz serwer. Zrobimy jeszcze raz tylko w notatniku.

In [9]:
import subprocess, time

server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [18]:
import requests
response = requests.get("http://localhost:5000/")
response

response = requests.get("http://localhost:5000/")
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")

Status: 200
Treść: Witaj w systemie monitoringu transakcji!


In [5]:
server.kill() # Zabijamy server

NameError: name 'server' is not defined

In [3]:
%%file app.py
from flask import Flask

app = Flask(__name__)

# ROUTING - sciezki do ktorych użytkownik ma dostęp
@app.route("/")                      # adres URL: http://localhost:5000/
def home():
    return "Witaj w systemie monitoringu transakcji!"

@app.route("/hello")
def hi():
    return "Hello world"

if __name__ == "__main__":              # Funkcja nie zadziała, jeżli kod zostanie zaimportowany
    app.run(host="0.0.0.0", port=5000)  # Uruchamianie na konkretnych porcie

Overwriting app.py


In [ ]:
server.kill()

In [ ]:
import subprocess, time

server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [6]:
import requests
response = requests.get("http://localhost:5000/hello")
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")

ConnectionError: HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /hello (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fd958482c10>: Failed to establish a new connection: [Errno 111] Connection refused'))

Pamiętać o stworzeniu servera jeżeli pracujemy tylko w notatniku

In [20]:
server.kill()

In [9]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Witaj w systemie monitoringu transakcji!"

@app.route("/hello")                  # GET /hello?name=Anna
def hello():
    name = request.args.get("name", "nieznajomy")   # odczytaj parametr query
    return f"Cześć, {name}!"                        # nieznajomy jeżeli słowo kluczowe nie wystąpiło w adresie

@app.route("/suma")
def suma():
    a = float(request.args.get("a", 0))
    b = request.args.get("b", 0, type=float)
    return f"Suma: {a+b}"

@app.route("/transaction/<tx_id>")    # GET /transaction/TX0042
def get_transaction(tx_id): # Przakazywanie parametru z <>
    return jsonify({"tx_id": tx_id, "status": "znaleziono"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [10]:
server.kill()

In [11]:
import subprocess, time
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [12]:
server

<Popen: returncode: None args: ['python', 'app.py']>

In [13]:
import requests
response = requests.get("http://localhost:5000/hello?name=alek")
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")

Status: 200
Treść: Cześć, alek!


In [14]:
response = requests.get("http://localhost:5000/suma?a=10&b=5")
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")

Status: 200
Treść: Suma: 15.0


In [15]:
response = requests.get("http://localhost:5000/transaction/TX0042")
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")

Status: 200
Treść: {"status":"znaleziono","tx_id":"TX0042"}



In [31]:
server.kill()

In [32]:
server

<Popen: returncode: 1 args: ['python', 'app.py']>

Można wykorzystać do wyszukiwania klienta

W większości przypadków nie podajemy pojedynczych argumentów - motoda `post` zamiast `get`

Restartujemy kernel

In [1]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/echo", methods=["POST"])  # akceptuj tylko POST
def echo():
    data = request.get_json()           # odczytaj ciało JSON
    return jsonify({
        "otrzymalem": data,
        "liczba_pol": len(data),
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [35]:
server.terminate()
server.wait()

1

In [2]:
import subprocess, time
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [4]:
import requests

dane = {"a": 5, "b": 45}

response = requests.post("http://localhost:5000/echo", json=dane)
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")
print(f"response.josn: {response.json()}")

Status: 200
Treść: {"liczba_pol":2,"otrzymalem":{"a":5,"b":45}}

response.josn: {'liczba_pol': 2, 'otrzymalem': {'a': 5, 'b': 45}}


In [ ]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/echo", methods=["POST"])  # akceptuj tylko POST
def echo():
    data = request.get_json()           # odczytaj ciało JSON
    return jsonify({
        "otrzymalem": data,
        "liczba_pol": len(data),
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
import requests

dane = {"a": 5, "b": 45}

response = requests.post("http://localhost:5000/echo", json=dane)
print(f"Status: {response.status_code}")
print(f"Treść: {response.text}")
print(f"response.josn: {response.json()}")